# Readme
This script serves as the supplementary numerical experiment based on the paper
https://arxiv.org/abs/2508.14803
to determine the minimum separation of the 2D Sobol sequence.

## Install libraries

In [ ]:
from numba import njit
import matplotlib.pyplot as plt
!pip install qmcpy --quiet
import numpy as np
import qmcpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 13.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done


## define auxiliary functions

In [ ]:
def float_to_base2representation(v):
    """
    represent (binary) rational v as
    v = w*2^{c}
    """
    if v == 0:  # Handle zero case
        return "0"
    w = int(v*2**20)
    c = -20
    while w%2 == 0:
        c += 1
        w //= 2

    str_expr = (str(w) + r" \cdot " if w != 1 else "") + "2^{" + str(c) + "}"
    binary_repr = str_expr.replace("^{","**(").replace("}",")").replace(r"\cdot","*")
    assert eval(binary_repr) == v # check the representation is correct
    return str_expr

def mydist(point1, point2):
    """
    define infinity norm independently for check
    """
    return np.max(np.abs(point1-point2))

## Find pair (p,q) which updates the minimum separation

In [ ]:
N = 2**12
points = qmcpy.Sobol(2, randomize="None").gen_samples(N)

"""
Find pair (p,q) which updates the minimum separation.
Distance is measured by the infinity norm.
"""
res = float("inf")
table = []
for i in range(1,len(points)):
    for j in range(i):
        sep = np.linalg.norm(points[i]-points[j], np.inf) / 2 # separation is half of distance
        if sep < res: # if updated the minimum
            table.append((j,i,sep))
            res = sep


/usr/local/lib/python3.12/dist-packages/qmcpy/discrete_distribution/digital_net_b2/digital_net_b2.py:421: ParameterWarning: Without randomization, the first digtial net point is the origin
  warnings.warn("Without randomization, the first digtial net point is the origin",ParameterWarning)


## Show the pair $(p,q)$ and distance and the minimum separation $\|x_p - x_q\|_\infty/2$

In [ ]:
print(f"{"p":>5} {"q":>5} {"separation":>16} {"separation (binary)"}")
for p,q,sep in table:
    assert mydist(points[p],points[q])/2 == sep # check the separation of x_p and x_q
    s = float_to_base2representation(sep)
    print(f"{p:>5} {q:>5} {sep:>16} {s}")

    p     q       separation separation (binary)
    0     1             0.25 2^{-2}
    1     2            0.125 2^{-3}
    2     4           0.0625 2^{-4}
    1     9          0.03125 2^{-5}
   24    36        0.0234375 3 \cdot 2^{-7}
   40    48         0.015625 2^{-6}
    8   112       0.00390625 2^{-8}
    1   129      0.001953125 2^{-9}
  504   516    0.00146484375 3 \cdot 2^{-11}
  640   768     0.0009765625 2^{-10}
 2176  3840   0.000244140625 2^{-12}


## Output table

In [ ]:
P = r"$p$ & "
Q = r"$q$ & "
DIST = r"$\|\bsx_p - \bsx_q\|_\infty$ & "
N = r"N & "
SEP = r"$q_\infty(Q_N)$ & "

for p,q,sep in table:
    n = q+1
    dist = sep*2
    assert dist == mydist(points[p],points[q])
    P += (r"$" + str(p)+ r"$" + " & ")
    Q += (r"$" + str(q)+ r"$" +" & ")
    DIST += (r"$" + float_to_base2representation(dist)+ r"$" +" & ")
    N += (r"$" + str(n)+ r"$" +" & ")
    SEP += (r"$" + float_to_base2representation(sep)+ r"$" +" & ")

for i in [P,Q,DIST,N,SEP]:
    print(r"\hline")
    print(i + r"\\")



\hline
$p$ & $0$ & $1$ & $2$ & $1$ & $24$ & $40$ & $8$ & $1$ & $504$ & $640$ & $2176$ & \\
\hline
$q$ & $1$ & $2$ & $4$ & $9$ & $36$ & $48$ & $112$ & $129$ & $516$ & $768$ & $3840$ & \\
\hline
$\|\bsx_p - \bsx_q\|_\infty$ & $2^{-1}$ & $2^{-2}$ & $2^{-3}$ & $2^{-4}$ & $3 \cdot 2^{-6}$ & $2^{-5}$ & $2^{-7}$ & $2^{-8}$ & $3 \cdot 2^{-10}$ & $2^{-9}$ & $2^{-11}$ & \\
\hline
N & $2$ & $3$ & $5$ & $10$ & $37$ & $49$ & $113$ & $130$ & $517$ & $769$ & $3841$ & \\
\hline
$q_\infty(Q_N)$ & $2^{-2}$ & $2^{-3}$ & $2^{-4}$ & $2^{-5}$ & $3 \cdot 2^{-7}$ & $2^{-6}$ & $2^{-8}$ & $2^{-9}$ & $3 \cdot 2^{-11}$ & $2^{-10}$ & $2^{-12}$ & \\


## Check that the theorem is correct

In [ ]:
"""
calculate separation for the points given in Section~4,
for each case (i), ..., (iv)
"""

def case1(v):
    m = 2**v
    p = 1
    q = 2**(2**v - 1) + 1
    p1, p2 = 2**(-1), 2**(-1)
    q1, q2 = 2**(-1) + 2**(-2**v), 2**(-1) - 2**(-2**v)
    assert np.array_equal(points[p], (p1, p2)), (points[p], p1, p2)
    assert np.array_equal(points[q], (q1, q2)), (points[q], q1, q2)
    return p,q,2**(-2**v - 1)

def case2(v):
    p = 2**(2**(v-1) - 1)
    q = 2**(2**v - 1) - 2**(2**(v-1))
    VV = 2**(v-1)
    p1, p2 = 2**(-VV), 1-2**(-VV)
    q1, q2 = 2**(-VV) - 2**(-2**v+1), 1 - 2**(-VV) - 2**(-2**v+1)
    assert np.array_equal(points[p], (p1, p2)), (points[p], p1, p2)
    assert np.array_equal(points[q], (q1, q2)), (points[q], q1, q2)
    return p,q,2**(-2**v)

def case3(v):
    p = 2**(2**v + 1) - 8
    q = 2**(2**v + 1) + 4
    p1, p2 = 2**(-3) - 2**(-(2**v)-1), 2**(-2) + 2**(-3) + 2**(-2**v) + 2**(-2**v-1)
    q1, q2 = 2**(-3) + 2**(-(2**v)-2), 2**(-2) + 2**(-3) + 2**(-2**v-1) + 2**(-2**v-2)
    assert np.array_equal(points[p], (p1, p2)), (points[p], p1, p2)
    assert np.array_equal(points[q], (q1, q2)), (points[q], q1, q2)
    return p,q,3*2**(-2**v - 3)

def case4(v,w):
    p = 2**(2**v + 2**w - 1) + 2**(2**v - 1)
    q = 2**(2**v + 2**w) - 2**(2**v)
    p1, p2 = 2**(-2**v) + 2**(-2**v - 2**w), 2**(-2**w) - 2**(-2**v - 2**w)
    q1, q2 = 2**(-2**v) - 2**(-2**v - 2**w), 2**(-2**w) + 2**(-2**v - 2**w)
    assert np.array_equal(points[p], (p1, p2)), (points[p], p1, p2)
    assert np.array_equal(points[q], (q1, q2)), (points[q], q1, q2)
    return p,q,2**(-2**v - 2**w)

def separation_of_two_points(p,q):
    return np.linalg.norm(points[p]-points[q], np.inf) / 2

assert separation_of_two_points(0,1) == np.max(np.abs(points[0]-points[1]))/2

updated_thm = [(0, 1, np.float64(0.25)), (1, 2, np.float64(0.125))] # not covered by theorem
for v in [2,3,4]:
    for case in [case1, case2, case3]:
        if v == 4 and case != case2:
           continue
        p,q,sep = result = case(v)
        assert separation_of_two_points(p,q) == sep, (p,q,sep)
        assert result in table
        updated_thm.append(result)

for v,w in [(2,1),(3,1),(3,2)]:
    p,q,sep = result = case4(v,w)
    assert separation_of_two_points(p,q) == sep
    assert result in table
    updated_thm.append(result)

updated_thm.sort(key=lambda x:x[1]) # sort by the value of q
assert sorted(table) == sorted(updated_thm)

print("Checked for the table:")
f"{table}"

Checked for the table:


'[(0, 1, np.float64(0.25)), (1, 2, np.float64(0.125)), (2, 4, np.float64(0.0625)), (1, 9, np.float64(0.03125)), (24, 36, np.float64(0.0234375)), (40, 48, np.float64(0.015625)), (8, 112, np.float64(0.00390625)), (1, 129, np.float64(0.001953125)), (504, 516, np.float64(0.00146484375)), (640, 768, np.float64(0.0009765625)), (2176, 3840, np.float64(0.000244140625))]'